### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [2]:
from unsloth import FastModel
from unsloth import FastLanguageModel

import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B",
    max_seq_length = 2048,
    load_in_4bit = True,
)



/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("gorkemergune/ayarlicazhocam_finetune", split = "train")

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [ ]:
dataset[100]

{'messages': [{'content': 'Do I need to code every day?',
   'images': None,
   'role': 'user',
   'thinking': None,
   'tool_calls': None},
  {'content': "There's a popular idea: 'code every day, do 100 Days of Code.' But the truth is: quality > frequency. Writing meaningless code for 10 minutes every day is less valuable than working focused on a project for 2 hours three times a week. Consistency matters but you don't need to be obsessive. Some days learn, read, research — that's growth too. Some days rest — the brain works better when rested. Gorkem probably doesn't code every day but looking at his projects there's a constant productivity. The key is finding a sustainable pace.",
   'images': None,
   'role': 'assistant',
   'thinking': 'Daily coding myth, quality vs frequency, sustainability.',
   'tool_calls': None}],
 'language': 'en'}

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [ ]:
dataset[0]

{'messages': [{'content': 'Who developed you?',
   'images': None,
   'role': 'user',
   'thinking': None,
   'tool_calls': None},
  {'content': "I'm the AI assistant of the ayarlicazhocam platform. This platform was developed by Gorkem Ergune to guide people in engineering, software, and university exam preparation. I'm the digital assistant of this platform — here to help with your questions, provide project information, and offer career guidance.",
   'images': None,
   'role': 'assistant',
   'thinking': 'Who developed question, ayarlicazhocam, Gorkem Ergune.',
   'tool_calls': None}],
 'language': 'en'}

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["messages"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [ ]:
dataset[100]["text"]

"<|begin_of_text|><start_of_turn>user\nDo I need to code every day?<end_of_turn>\n<start_of_turn>model\nThere's a popular idea: 'code every day, do 100 Days of Code.' But the truth is: quality > frequency. Writing meaningless code for 10 minutes every day is less valuable than working focused on a project for 2 hours three times a week. Consistency matters but you don't need to be obsessive. Some days learn, read, research — that's growth too. Some days rest — the brain works better when rested. Gorkem probably doesn't code every day but looking at his projects there's a constant productivity. The key is finding a sustainable pace.<end_of_turn>\n"

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",

        fp16=True,
        bf16=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/429 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes! Unsloth now auto-detects the instruction and response parts from the tokenizer's chat template, so we don't need to pass `instruction_part` and `response_part` anymore. You can still pass them explicitly if you use a custom chat template.

In [ ]:
print(type(trainer.data_collator))
print(type(model))
print(type(tokenizer))

<class 'trl.trainer.sft_trainer.DataCollatorForLanguageModeling'>
<class 'peft.peft_model.PeftModelForCausalLM'>
<class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<end_of_turn>\n<start_of_turn>user\n' and response_part = '<end_of_turn>\n<start_of_turn>model\n'


Map:   0%|          | 0/429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Unsloth: Removed 381 out of 429 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [ ]:
tokenizer.decode(trainer.train_dataset[30]["input_ids"])

"<|begin_of_text|><|begin_of_text|><start_of_turn>user\nmotivasyonum düştü ders çalışamıyorum<end_of_turn>\n<start_of_turn>model\nMotivasyon düşüklüğü çok normal, herkese oluyor. Ama şunu bil: motivasyon gelip geçici bir şey, disiplin kalıcıdır. Her gün motive olmanı bekleme, bazı günler isteksiz olacaksın ama yine de oturup 30 dakika bile çalışırsan alışkanlık oluşur. Birkaç taktik: büyük hedefleri küçük parçalara böl. 'Bugün 5 saat çalışacağım' demek yerine 'bugün sadece 3 tane matematik sorusu çözeceğim' de. Başladıktan sonra devam etmek daha kolay. Çalışma ortamını değiştir — kütüphaneye git, farklı bir odada çalış. Telefonunu başka odaya bırak. Ve en önemlisi kendini başkalarıyla kıyaslama. Herkesin kendi temposu var. Kötü geçen bir gün seni tanımlamaz. Yarın yeni bir gün, kalk ve devam et.<end_of_turn>\n"

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[30]["labels"]]).replace(tokenizer.pad_token, " ")

"                                Motivasyon düşüklüğü çok normal, herkese oluyor. Ama şunu bil: motivasyon gelip geçici bir şey, disiplin kalıcıdır. Her gün motive olmanı bekleme, bazı günler isteksiz olacaksın ama yine de oturup 30 dakika bile çalışırsan alışkanlık oluşur. Birkaç taktik: büyük hedefleri küçük parçalara böl. 'Bugün 5 saat çalışacağım' demek yerine 'bugün sadece 3 tane matematik sorusu çözeceğim' de. Başladıktan sonra devam etmek daha kolay. Çalışma ortamını değiştir — kütüphaneye git, farklı bir odada çalış. Telefonunu başka odaya bırak. Ve en önemlisi kendini başkalarıyla kıyaslama. Herkesin kendi temposu var. Kötü geçen bir gün seni tanımlamaz. Yarın yeni bir gün, kalk ve devam et.<end_of_turn>\n"

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.051 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.927700
2,2.469000
3,2.602500
4,2.659800
5,2.487100
6,2.782400
7,2.318700
8,2.506300
9,2.477300
10,2.604100


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

123.1833 seconds used for training.
2.05 minutes used for training.
Peak reserved memory = 3.051 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 20.95 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)
outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

["<|begin_of_text|><start_of_turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<end_of_turn>\n<start_of_turn>model\nAh, yes, I see what you mean. The sequence we're talking about is called the Fibonacci sequence, and it's defined as follows:\nF0 = 1, F1 = 1, Fn = Fn-1 + Fn-2 for n >= 2\nSo, for example, F3 ="]

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Great question, user! The sky is made up of gases like oxygen, nitrogen, and water vapour. When sunlight shines through these gases, it bounces off them and gives them their colour. If you're curious, the reason the Sun looks yellow is because it's made up of hot gas. When it shines


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("Llama_3_lora")  # Local saving
tokenizer.save_pretrained("Llama_3_lora")

('Llama_3_lora/tokenizer_config.json',
 'Llama_3_lora/special_tokens_map.json',
 'Llama_3_lora/chat_template.jinja',
 'Llama_3_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "Llama-LoRa", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Who is Gorkem?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Hey there! Gorkem is a senior developer with a decade of experience, mainly focused on web development with frameworks like React, React Native, Vue, and Angular. His portfolio showcases a variety of projects like React components, e-commerce applications, and APIs. Gorkem also has experience with Node.js, GraphQL,


In [ ]:
import os

cache = os.path.expanduser("~/.cache/huggingface/hub")
print(cache)

for root, dirs, files in os.walk(cache):
    print(root, len(files))
    break